# Demo — Load Model, Run Inference, Visualize Performance

This notebook is a runnable demo for the repository. It:

1. Loads saved evaluation artifacts (metrics + predictions) and creates interpretability plots.
2. Loads a trained **NER transformer checkpoint** (if present) and runs inference on example sentences.
3. Shows qualitative error slices from saved test predictions.

Prerequisites:
- Install dependencies: `pip install -r requirements.txt`
- Generate artifacts by running a pipeline (from repo root):
  - Quick: `bash scripts/run_quick_pipeline.sh`
  - Full: `bash scripts/run_full_pipeline.sh`

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from transformers import AutoModelForTokenClassification, AutoTokenizer


def find_project_root(start: Path) -> Path:
    """Find the repository root by walking upward until src/medical_ai_project exists."""
    current = start.resolve()
    for parent in [current, *current.parents]:
        if (parent / 'src' / 'medical_ai_project').exists():
            return parent
    raise FileNotFoundError('Could not locate project root (expected src/medical_ai_project).')


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

sns.set_theme(style='whitegrid')
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
from medical_ai_project.data.pubmed_rct20k import simple_tokenize
from medical_ai_project.evaluation.analysis import extract_representative_examples, summarize_error_modes

ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
NER_METRICS_PATH = ARTIFACTS_ROOT / 'transformer' / 'metrics' / 'test_metrics.json'
NER_REPORT_PATH = ARTIFACTS_ROOT / 'transformer' / 'metrics' / 'entity_report.json'
NER_PRED_PATH = ARTIFACTS_ROOT / 'transformer' / 'predictions' / 'test_predictions.csv'

for p in [NER_METRICS_PATH, NER_REPORT_PATH, NER_PRED_PATH]:
    print(p, 'exists=' + str(p.exists()))

## NER performance from saved artifacts

These metrics are saved by the transformer training pipeline under `artifacts/transformer/`.
We plot per-entity F1 to make performance differences interpretable by entity type.

In [ ]:
if not NER_METRICS_PATH.exists():
    raise FileNotFoundError('Missing NER metrics. Run `bash scripts/run_quick_pipeline.sh` first.')

ner_metrics = json.loads(NER_METRICS_PATH.read_text(encoding='utf-8'))
ner_report = json.loads(NER_REPORT_PATH.read_text(encoding='utf-8'))

pd.DataFrame([ner_metrics])

In [ ]:
report_rows = []
for entity_type, stats in ner_report.items():
    report_rows.append({'entity': entity_type, **stats})
report_df = pd.DataFrame(report_rows).sort_values('support', ascending=False)
report_df

In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=report_df, x='entity', y='f1')
ax.set_title('NER Transformer: Entity-level F1 by Type')
ax.set_xlabel('Entity type')
ax.set_ylabel('F1')
ax.set_ylim(0.0, 1.0)
plt.tight_layout()
plt.show()

## Load the trained NER checkpoint and run inference

The pipeline saves a Hugging Face checkpoint under `artifacts/transformer/checkpoints/` (folders like `checkpoint-*/`).
We load it and run inference on a few sentences for an end-to-end demo.

In [ ]:
CHECKPOINTS_ROOT = ARTIFACTS_ROOT / 'transformer' / 'checkpoints'
candidates = list(CHECKPOINTS_ROOT.glob('checkpoint-*')) if CHECKPOINTS_ROOT.exists() else []
def _step(p: Path) -> int:
    try:
        return int(p.name.split('-')[-1])
    except Exception:
        return -1
candidates = sorted(candidates, key=_step)
if not candidates:
    raise FileNotFoundError(
        f'No transformer checkpoints found under {CHECKPOINTS_ROOT}. Run `bash scripts/run_quick_pipeline.sh` first.'
    )
CHECKPOINT_DIR = candidates[-1]
print('Using checkpoint:', CHECKPOINT_DIR)

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
model = AutoModelForTokenClassification.from_pretrained(CHECKPOINT_DIR)
model.eval()
id2label = {int(k): v for k, v in model.config.id2label.items()}
len(id2label), list(id2label.items())[:5]

In [ ]:
def ner_predict_tokens(text: str, max_length: int = 128) -> pd.DataFrame:
    """Run token-level NER inference and return a tidy token/tag table."""
    tokens = simple_tokenize(text)
    if not tokens:
        return pd.DataFrame({'token': [], 'pred_tag': []})

    encoded = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
    )

    with torch.no_grad():
        outputs = model(**encoded)
        pred_ids = outputs.logits.argmax(dim=-1).squeeze(0).cpu().numpy().tolist()

    # Align subword predictions back to word-level tokens by taking the first subword prediction.
    word_ids = encoded.word_ids(batch_index=0)
    word_level = []
    prev_word_id = None
    for token_pred_id, word_id in zip(pred_ids, word_ids):
        if word_id is None:
            continue
        if word_id == prev_word_id:
            continue
        word_level.append((tokens[word_id], id2label[int(token_pred_id)]))
        prev_word_id = word_id

    return pd.DataFrame(word_level, columns=['token', 'pred_tag'])

In [ ]:
examples = [
    'Aspirin reduced the risk of stroke in patients with hypertension.',
    'Magnetic resonance imaging was performed before surgery to evaluate the brain.',
    'Metformin was compared with insulin for diabetes management.',
]

for text in examples:
    display(ner_predict_tokens(text).head(60))

## Qualitative error analysis from saved predictions

We use the saved test predictions to surface representative correct/error examples and summarize common error modes.

In [ ]:
pred_df = pd.read_csv(NER_PRED_PATH)
pred_df.head()

In [ ]:
examples = extract_representative_examples(pred_df, max_correct=5, max_errors=5)
pd.DataFrame(examples['correct_examples'])

In [ ]:
pd.DataFrame(examples['error_examples'])

In [ ]:
summary = summarize_error_modes(pred_df, top_k=15)
pd.DataFrame(summary['top_confusions'])

## (Optional) Classification performance visualization

This section visualizes sentence classification metrics from existing artifacts (if present).

In [ ]:
CLS_METRICS_PATH = ARTIFACTS_ROOT / 'classification_lstm' / 'metrics' / 'test_metrics.json'
CLS_CM_PATH = ARTIFACTS_ROOT / 'classification_lstm' / 'metrics' / 'confusion_matrix.csv'

if CLS_METRICS_PATH.exists() and CLS_CM_PATH.exists():
    cls_metrics = json.loads(CLS_METRICS_PATH.read_text(encoding='utf-8'))
    display(pd.DataFrame([{'accuracy': cls_metrics['accuracy'], 'f1_macro': cls_metrics['f1_macro'], 'f1_weighted': cls_metrics['f1_weighted']}]))

    cm = pd.read_csv(CLS_CM_PATH)
    labels = cm['true_label'].tolist()
    matrix = cm.drop(columns=['true_label']).to_numpy()

    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    ax.set_title('Classification (LSTM): Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.tight_layout()
    plt.show()
else:
    print('Classification artifacts not found. Run a pipeline to generate them.')